<a href="https://colab.research.google.com/github/Vivisteria11/AI-MathAgent/blob/main/Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')

In [2]:
import os
from google.colab import userdata

os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')

In [3]:
!pip install langchain langchain-community
!pip install langchain-qdrant
!pip install agno duckduckgo-search
from agno.agent import Agent #AGENT
from agno.models.google import Gemini #llm
from agno.tools.duckduckgo import DuckDuckGoTools #provider





In [4]:
!pip install langchain-google-genai


In [6]:
import os

os.environ["USER_AGENT"] = "RakshitaMathAgent/1.0 (langchain)"

In [7]:
# Retrieval
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Vector Database
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
# Augmentation
from langchain_core.prompts import PromptTemplate
# Generation
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain.schema import StrOutputParser

In [ ]:
!pip install datasets
from datasets import load_dataset
!pip install --upgrade datasets fsspec pyarrow





## Dataset Loaded ,Grade School Math Dataset

In [ ]:
from datasets import load_dataset

gsm_dataset = load_dataset("gsm8k", "main")

In [ ]:
from langchain.schema import Document
documents = [
    Document(page_content=f"Q: {item['question']}\nA: {item['answer']}")
    for item in gsm_dataset["train"]
]
print(documents)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=50
)

# Split into chunks
chunks = text_splitter.split_documents(documents)


print(f"Total chunks: {len(chunks)}")


In [8]:
import os
from google.colab import userdata

os.environ['QDRANT_URL']=userdata.get('QDRANT_URL')
os.environ['QDRANT_API_KEY']=userdata.get('QDRANT_API_KEY')

client = QdrantClient(
    url=os.environ['QDRANT_URL'],
    api_key=os.environ['QDRANT_API_KEY']
)



### **QDRANT VECTOR DB**

In [ ]:
collection_name = "gsm_chunks"


try:
    collection_info = client.get_collection(collection_name=collection_name)
    print(f"Collection '{collection_name}' already exists.")
except:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=1024,
            distance=Distance.COSINE
        )
    )


In [9]:
!pip install fastembed

In [10]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
embeddings = FastEmbedEmbeddings(model_name="thenlper/gte-large")

/usr/local/lib/python3.11/dist-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model thenlper/gte-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


In [ ]:
from langchain_community.vectorstores import Qdrant

qdrant_store = Qdrant.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=os.environ['QDRANT_URL'],
    api_key=os.environ['QDRANT_API_KEY'],
    collection_name=collection_name,
    force_recreate = True
)


To run Already existing vector DB

In [11]:
from langchain_qdrant import Qdrant
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from qdrant_client import QdrantClient


embeddings = FastEmbedEmbeddings(model_name="thenlper/gte-large")

client = QdrantClient(
    url=os.environ['QDRANT_URL'],
    api_key=os.environ['QDRANT_API_KEY']
)


qdrant_store = Qdrant(
    client=client,
    collection_name="gsm_chunks",
    embeddings=embeddings,
)



/usr/local/lib/python3.11/dist-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model thenlper/gte-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
/tmp/ipython-input-11-1284109104.py:14: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use :class:`~QdrantVectorStore` instead.
  qdrant_store = Qdrant(


In [12]:
retriever = qdrant_store.as_retriever()


In [13]:
docs = qdrant_store.similarity_search("How many apples are there if John eats 2 out of 20?")
for doc in docs:
    print(doc.page_content)


Q: Archibald eats 1 apple a day for two weeks. Over the next three weeks, he eats the same number of apples as the total of the first two weeks. Over the next two weeks, he eats 3 apples a day. Over these 7 weeks, how many apples does he average a week?
A: He ate 14 apples the first two weeks because 14 x 1 = 14
He ate 14 apples in the next three weeks because 14 = <<14=14>>14
He ate 42 apples in the final two weeks because 14 x 3 = <<14*3=42>>42
He ate 70 apples in total.
He averaged 10 apples a week because 70 / 7 = <<70/7=10>>10
#### 10
Q: Tim has 30 less apples than Martha, and Harry has half as many apples as Tim. If Martha has 68 apples, how many apples does Harry have?
A: Tim has 68-30 = <<68-30=38>>38 apples.
Harry has 38/2 = <<38/2=19>>19 apples.
#### 19
Q: Lexie and Tom went apple picking. Lexie picked 12 apples and Tom picked twice as many apples. How many apples did they collect altogether?
A: Tom picked 12 x 2 = <<12*2=24>>24 apples
Altogether, they picked 12 + 24 = <<12+2

In [84]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [85]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "query"],
    template="""
You are a helpful math tutor who uses real-world data and calculations to answer questions.If you dont find anything in the knowledge base,

below is a web search result and a question that may involve both factual information and math reasoning.

Search Result:
{context}

Question:
{query}

Answer:
""".strip()
)



In [86]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


### **Langchain for RAG**

In [87]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {
        "context": retriever,
        "query": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [18]:
#testing the chain
response = chain.invoke("A shop sells pencils at ₹5 each. If Rakshita buys 7 pencils, how much does she have to pay in total")
print(response)

Okay, I can help you calculate the total cost of the pencils Rakshita buys.

*   **Cost per pencil:** ₹5
*   **Number of pencils:** 7

To find the total cost, we multiply the cost per pencil by the number of pencils:

Total cost = ₹5/pencil \* 7 pencils = ₹35

**Answer:** Rakshita has to pay ₹35 in total.


In [ ]:
import os
from google.colab import userdata
os.environ["GUARDRAILS_HUB_API_KEY"] = userdata.get('GUARDRAILS_API_KEY')


In [19]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### **INPUT GUARDRAIL**

In [22]:
chain = (
    {
        "context": retriever,
        "query": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [115]:
import os
from google.colab import userdata

os.environ["SAMBANOVA_API_KEY"] = userdata.get("SAMBANOVA_API_KEY")


In [116]:
from openai import OpenAI

sambanova = OpenAI(
    api_key=os.environ["SAMBANOVA_API_KEY"],
    base_url="https://api.sambanova.ai/v1",
)


In [115]:
from langchain_core.runnables import RunnableLambda

def math_question(query: str) -> str:
    prompt = f"""Classify the following question strictly.

Question: "{query}"

Does it either:Involve solving or explaining a math problem OR
Relate to general mathematical topics like awards, education policies, competitions or similar current affairs?

Respond with exactly one word: Yes or No."""

    try:
        result = llm.invoke(prompt)
        answer = result.content.strip().lower()
        if  answer == "yes":
           return query

        return "Sorry, I can only help with math questions right now."
    except Exception as e:
        print(e)
        return "Sorry, I can only help with math questions right now."

In [113]:
def length(query: str) -> str:
    if len(query) <= 500:
        return query
    return " Question exceeds 500 characters."

In [54]:

validate_math = RunnableLambda(math_question)
validate_length = RunnableLambda(length)


input_guard = validate_length | validate_math


In [55]:
#wrapped the guard in runnable template from langchain
chain = (
    {
        "context": retriever,
        "query": RunnablePassthrough() | input_guard
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [57]:
result = math_question("What is an animal?")
print(result)


Sorry, I can only help with math questions right now.


In [56]:
res = chain.invoke("What is Iguana?")
print(res)

Okay, I understand. I am ready for your math question. Please ask away! I will do my best to help you using real-world data and calculations.


## **Output Guardrail**

In [49]:
def math_output(answer):

    if hasattr(answer, "content"):
        answer = answer.content
    vague = ["i don't know", "not sure", "maybe", "unsure", "cannot answer"]
    if not answer.strip() or any(x in answer.lower() for x in vague):
        return "I'm not confident enough to answer this question right now."
    prompt = f"""
    Is this answer confident and math-related either as an  explanation  to a problem or a general fact)?
    "{answer}"
    Just reply Yes or No."""


    result = llm.invoke(prompt)
    verdict = result.content.lower().strip() if hasattr(result, "content") else result.lower().strip()

    return answer.strip() if "yes" in verdict else "I'm not confident enough to answer this question right now."



In [50]:

output_guard = RunnableLambda(math_output)

In [88]:
chain = (
    {
        "context": retriever,
        "query": RunnablePassthrough() | input_guard
    }
    | prompt
    | llm
    | output_guard
    | StrOutputParser()
)


In [36]:
response = chain.invoke("If the function f(x) = x³ - 3x + 1 has a local maximum and a local minimum, find the distance between those two points.")
print(response)

Okay, this is an interesting problem that combines calculus concepts with finding a distance. Here's how we can approach it:

**1. Find the critical points:**

*   To find local maxima and minima, we need to find the critical points of the function. These are the points where the derivative is either zero or undefined.
*   First, find the derivative of f(x):
    f'(x) = 3x² - 3

*   Set the derivative equal to zero and solve for x:
    3x² - 3 = 0
    3x² = 3
    x² = 1
    x = ±1

*   So, the critical points are x = 1 and x = -1.

**2. Determine if the critical points are local maxima or minima:**

*   We can use the second derivative test to determine the nature of the critical points.
*   Find the second derivative of f(x):
    f''(x) = 6x

*   Evaluate the second derivative at each critical point:
    *   f''(1) = 6(1) = 6. Since f''(1) > 0, x = 1 is a local minimum.
    *   f''(-1) = 6(-1) = -6. Since f''(-1) < 0, x = -1 is a local maximum.

**3. Find the y-coordinates of the loca

In [33]:
llm.invoke("What is the capital of France?")


AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--9794f619-a81f-4fb7-bd92-2961085eed34-0', usage_metadata={'input_tokens': 7, 'output_tokens': 8, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})

In [59]:
response = chain.invoke("Tell me something interesting about clouds.")
print(response)


Okay, I understand. I can only help with math questions. Do you have a math question you would like me to help you with? I can use the information in the search results to help, or I can try to answer a new question. Just let me know what you need.


In [60]:
response = chain.invoke("On Monday, Fred ate 3 cookies. On Tuesday, Fred ate 6 cookies. On Wednesday, Fred ate 9 cookies. On Thursday, Fred ate 12 cookies. On Friday, Fred ate 15 cookies. How many cookies did Fred eat this week?")
print(response)


Okay, I can help you figure out how many cookies Fred ate this week.

*   **Monday:** 3 cookies
*   **Tuesday:** 6 cookies
*   **Wednesday:** 9 cookies
*   **Thursday:** 12 cookies
*   **Friday:** 15 cookies

To find the total, we add the number of cookies he ate each day:

3 + 6 + 9 + 12 + 15 = 45

**Answer:** Fred ate 45 cookies this week.


### **Websearch**

In [ ]:
!pip install guardrails-ai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.

In [ ]:
#tried implementing AI agent using AGNO but it caused issues between google packages hence removed
!pip install agno

In [61]:
from agno.models.google import Gemini
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.agent import Agent
llm = Gemini()
agent = Agent(model=llm)

In [62]:
tool = DuckDuckGoTools()

In [63]:
ai_agent = Agent(
    name = "Math Agent",
    model = llm,
    tools = [DuckDuckGoTools()],
    show_tool_calls=True,
    markdown = True,
    system_message = "You are an Expert Math tutor who is searching the web for answers not found in the knowledge Base"
)

In [64]:
#testing for duckduck go
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
query = "Who won the Fields Medal in 2022?"
results = search_tool.run(query)
print(results)


The Fields Medal is awarded (traditionally to mathematicians under the age of 40) to recognise outstanding mathematical achievement for existing work and for the promise of future achievement. Ukrainian mathematician (and LMS Honorary Member) Maryna Viazovska is only the second woman to win a Fields Medal, after Maryam Mirzakhani, who won in 2014. Maryam Mirzakhani was the first woman to win the Fields Medal (2014). In 2022 Maryna Viazovska, who was born in Ukraine, became the second woman to win; she received the award in Finland after that year's ceremony was moved from its original location in Russia because of Russia's invasion of Ukraine. List of Fields Medal winners from every year the award has been given out. All Fields Medal winners are listed below in order of popularity, but can be sorted by any column. People who won the Fields Medal award are listed along with photos for every Fields Medal winner that has a picture... This list ofFields Medal winners by university affiliat

In [65]:
fallback = RunnableLambda(lambda input: search_tool.run(input))

In [77]:
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

relevance_checker_prompt = PromptTemplate.from_template("""
Is the following document context useful to answer the question ,verify carefully based on if the question asked involves information from current affairs? Reply with only "yes" or "no".

Question: {question}

Context:
{context}
""")

relevance_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.3
)

def should_fallback_llm(question, docs):
    for doc in docs:
        prompt_input = {
            "question": question,
            "context": doc.page_content[:1000]
        }
        result = relevance_llm.invoke(relevance_checker_prompt.format(**prompt_input))
        verdict = result.content.strip().lower() if hasattr(result, "content") else result.strip().lower()
        if "yes" in verdict:
            return False
    return True

In [78]:
fallback_reason = RunnableLambda(
    lambda query: retriever.invoke(query)
    if not should_fallback_llm(query, retriever.invoke(query))
    else [Document(page_content=search_tool.run(query))]
)



In [79]:
import re

def format_math_output(response):
    if hasattr(response, "text"):
        content = response.text
    elif hasattr(response, "content"):
        content = response.content
    else:
        content = str(response)


    content = content.strip()
    content = content.replace("$$", "")
    content = content.replace("$", "")
    content = content.replace("**", "")
    content = re.sub(r"\\boxed{([^}]+)}", r"\1", content)
    content = re.sub(r"\n{3,}", "\n\n", content)

    return content


In [96]:
#this is the chain with fallback
from langchain_core.documents import Document

chain_with_fallback = (
    {
        "context": fallback_reason,
        "query": RunnablePassthrough() | input_guard
    }
    | prompt
    | llm
    | output_guard
    | StrOutputParser()
    | RunnableLambda(format_math_output)

)

In [94]:
response = chain_with_fallback.invoke("What is the current price of gold per gram in India, and how much is 10 grams?")

In [95]:
print(response)

Okay, I can help you with that!

Based on the provided search result, the current gold price per gram in India is ₹9,195.41 Indian Rupees (INR).

To calculate the price of 10 grams of gold, we simply multiply the price per gram by 10:

₹9,195.41/gram * 10 grams = ₹91,954.10

Therefore, 10 grams of gold currently costs ₹91,954.10 in India.


In [ ]:
print(repr(chain.invoke({"query": "Solve: sum of three consecutive squares is 365"})))



"Okay, I can help you solve this problem. Here's how we can approach it:\n\n**Understanding the Problem**\n\nWe need to find three consecutive numbers. If we square each of them and add the squares together, the total must equal 365.\n\n**Setting up the Equation**\n\nLet's use algebra:\n\n*   Let 'x' be the first number.\n*   The next consecutive number is 'x + 1'.\n*   The number after that is 'x + 2'.\n\nThe sum of their squares can be written as:\n\nx² + (x + 1)² + (x + 2)² = 365\n\n**Solving the Equation**\n\n1.  Expand the squared terms:\n\nx² + (x² + 2x + 1) + (x² + 4x + 4) = 365\n\n2.  Combine like terms:\n\n3x² + 6x + 5 = 365\n\n3.  Subtract 365 from both sides to set the equation to zero:\n\n3x² + 6x - 360 = 0\n\n4.  Divide the entire equation by 3 to simplify:\n\nx² + 2x - 120 = 0\n\n5.  Factor the quadratic equation:\n\n(x + 12)(x - 10) = 0\n\n6.  Solve for x:\n\nx + 12 = 0  or  x - 10 = 0\n\nx = -12  or  x = 10\n\n**Finding the Consecutive Numbers**\n\nWe have two possible 

In [61]:
response = ai_agent.run("What was the GDP growth rate of India in 2024?")
print(response)


RunResponse(content="Based on the search results, here's the information on India's GDP growth rate in 2024-25:\n\n*   **Real GDP Growth:** Estimates range from 6.4% to 7% for the financial year 2024-25.\n*   **Q4 (Jan-Mar) FY 2024-25:** 7.4%\n", content_type='str', thinking=None, reasoning_content=None, messages=[Message(role='system', content='You are an Expert Math tutor who is searching the web for answers not found in the knowledge Base', name=None, tool_call_id=None, tool_calls=None, audio=None, images=None, videos=None, files=None, audio_output=None, image_output=None, thinking=None, redacted_thinking=None, provider_data=None, citations=None, reasoning_content=None, tool_name=None, tool_args=None, tool_call_error=None, stop_after_tool_call=False, add_to_agent_memory=True, from_history=False, metrics=MessageMetrics(input_tokens=0, output_tokens=0, total_tokens=0, audio_tokens=0, input_audio_tokens=0, output_audio_tokens=0, cached_tokens=0, cache_write_tokens=0, reasoning_tokens=0

In [92]:
response = chain_with_fallback.invoke("The sum of the squares of three consecutive natural numbers is 365. What are the numbers?")
print(response)

Okay, I can help you solve this problem using the information from the search result.

Understanding the Problem

We need to find three natural numbers that follow each other (like 1, 2, 3 or 10, 11, 12) where, if you square each number and add the squares together, the result is 365.

Using Algebra to Solve

Let's use the suggestion from the search result and represent the three consecutive natural numbers as:

*   *n* - 1
*   *n*
*   *n* + 1

According to the problem, the sum of their squares is 365. So we can write the equation:

( *n* - 1 )<sup>2</sup> + *n*<sup>2</sup> + ( *n* + 1 )<sup>2</sup> = 365

Now, let's expand and simplify the equation:

*n*<sup>2</sup> - 2*n* + 1 + *n*<sup>2</sup> + *n*<sup>2</sup> + 2*n* + 1 = 365

Combine like terms:

3*n*<sup>2</sup> + 2 = 365

Subtract 2 from both sides:

3*n*<sup>2</sup> = 363

Divide both sides by 3:

*n*<sup>2</sup> = 121

Take the square root of both sides:

*n* = 11 (We only consider the positive root since we're looking for nat

In [80]:
response = chain_with_fallback.invoke("What was the GDP growth rate of India in 2024, and if the same rate continues, what will be India’s estimated GDP in 2025 assuming the 2023 GDP was $3.73 trillion?")
print(response)

Okay, I can help you with that! Based on the provided document, let's break down the calculation:

1. GDP Growth Rate in 2024:

*   The document states the Real GDP growth rate for 2023-24 was 8.2%. However, the document also states the Real GDP growth rate for 2024-25 is estimated at 6.4%. Since the question specifies 2024, I will use 8.2% to calculate the estimated GDP in 2025.

2. Calculate India's Estimated GDP in 2025:

*   2023 GDP: 3.73 trillion
*   Growth Rate: 8.2%

To calculate the estimated GDP in 2024, we'll apply the growth rate to the 2023 GDP:

*   Growth Amount: 3.73 trillion * 0.082 = 0.30586 trillion
*   Estimated 2024 GDP: 3.73 trillion + 0.30586 trillion = 4.03586 trillion

Now, to calculate the estimated GDP in 2025, assuming the same 8.2% growth rate continues:

*   Growth Amount: 4.03586 trillion * 0.082 = 0.33094 trillion
*   Estimated 2025 GDP: 4.03586 trillion + 0.33094 trillion = 4.3668 trillion

Answer:

*   The GDP growth rate of India in 2024 was 8.2%.
*  

In [ ]:
response =chain_with_fallback.invoke("What is the current population of India in 2024, and if 30% of them are students, how many students are there?")

In [ ]:
print(response)

Okay, I can help you with that.

I don't have access to a real-time, up-to-the-minute population count for India. Population figures are constantly changing. However, I can provide an estimate and then calculate the number of students based on that estimate.

1. Estimate of India's Population in 2024:

A reasonable estimate for India's population in 2024 is around 1.44 billion people. Please note that this is an estimate, and the actual number may vary. For a precise figure, you can consult the Population Division of the United Nations Department of Economic and Social Affairs or the World Bank.

2. Calculation of the Number of Students:

*   Percentage of students: 30%
*   Estimated total population: 1.44 billion

To find the number of students, we need to calculate 30% of 1.44 billion:

Number of students = 0.30 * 1,440,000,000 = 432,000,000

Answer:

Based on the estimated population of 1.44 billion for India in 2024, and assuming 30% are students, there would be approximately 432 m

In [ ]:
response = chain_with_fallback.invoke("Alpha and Beta are the roots of a quadratic equation:Ex squared minus six times x plus k equals zero.It is given that the sum of the squares of Alpha and Beta is equal to twenty.Find the value of k.")
print(response)

Let the quadratic equation be x^2 - 6x + k = 0.
Let \alpha and \beta be the roots of the equation.
By Vieta's formulas, we have:
\alpha + \beta = 6
\alpha \beta = k
We are given that \alpha^2 + \beta^2 = 20.
We know that (\alpha + \beta)^2 = \alpha^2 + \beta^2 + 2\alpha\beta.
Substituting the given values, we have:
(6)^2 = 20 + 2k
36 = 20 + 2k
36 - 20 = 2k
16 = 2k
k = \frac{16}{2}
k = 8

Final Answer: The final answer is 8


In [ ]:
response = chain_with_fallback.invoke("Who won the Fields Medal in 2022?")
print(response)

This question does not require math to solve.

Answer:
The Fields Medal in 2022 was awarded to Hugo Duminil-Copin, June Huh, James Maynard, and Maryna Viazovska.


### Human in the Loop

In [98]:
feedback = RunnableLambda(lambda inputs: feedback_loop(inputs["answer"], inputs["query"]))


In [100]:
def feedback_retry(answer: str, question: str):
    print("Question:", question)
    print("Answer:", answer)
    feedback = input("Was this helpful? (YES/NO): ").strip().lower()

    if feedback == "yes":
        return "Thanks I am glad it helped."
    else:
        print("Will reprompt the LLM for a better suited answer")
        retry_prompt = f"""Make this answer more simple ,easily understandable and clear to the student:
            Question: {question}
            Answer: {answer}
            Better Answer:"""

        improved = llm.invoke(retry_prompt)
        improved_answer = improved.content.strip() if hasattr(improved, "content") else str(improved)
        return format_math_output(improved_answer)

In [112]:
def feedback_chain(query):
    answer = chain_with_fallback.invoke(query)
    return feedback_retry(answer, query)

In [111]:
feedback = RunnableLambda(feedback_chain)

## **LangGraph Math Agent**

In [102]:
!pip install langgraph langchain langchain-core


In [103]:
from typing import TypedDict
from langgraph.graph import StateGraph
from langchain_core.runnables import RunnableLambda


In [104]:
class AgentState(TypedDict):
    query: str
    answer: str

In [105]:
def feedback_chain_fn(state: AgentState) -> AgentState:
    query = state["query"]
    answer = chain_with_fallback.invoke(query)
    improved = feedback_retry(answer, query)
    return {"query": query, "answer": improved}

In [106]:

feedback_node = RunnableLambda(feedback_chain_fn)


In [107]:
builder = StateGraph(AgentState)
builder.add_node("generate_with_feedback", feedback_node)
builder.set_entry_point("generate_with_feedback")
graph = builder.compile()



In [108]:

result = graph.invoke({"query": "Find the 10th term of an AP with sum formula 3n squared plus 2n"})
print(result["answer"])

Question: Find the 10th term of an AP with sum formula 3n squared plus 2n
Answer: Okay, I can help you find the 10th term of the arithmetic progression (AP) given the sum formula.

Understanding the Problem

We are given the sum of the first *n* terms of an AP as S_n = 3n² + 2n.  We need to find the 10th term, which we'll call a_10.

Using the Formula

The search result provides a crucial formula:  a_n = S_n - S_{n-1}.  This means the nth term is equal to the sum of the first n terms minus the sum of the first (n-1) terms.

Calculation

1.  Find S_10 (sum of the first 10 terms):
    S_10 = 3(10)² + 2(10) = 3(100) + 20 = 300 + 20 = 320

2.  Find S_9 (sum of the first 9 terms):
    S_9 = 3(9)² + 2(9) = 3(81) + 18 = 243 + 18 = 261

3.  Find a_10 (the 10th term):
    a_10 = S_10 - S_9 = 320 - 261 = 59

Answer:

The 10th term of the arithmetic progression is 59.
Was this helpful? (YES/NO): no
Will reprompt the LLM for a better suited answer
Okay, let's find the 10th number in this special n

In [109]:
res = feedback.invoke("The sum of the first n terms of a series is given by n times n plus one divided by two. What is the eighth term of the series?")
print(res)

Question: The sum of the first n terms of a series is given by n times n plus one divided by two. What is the eighth term of the series?
Answer: Okay, let's break this down. We're given a formula for the sum of the first *n* terms of a series:

S<sub>n</sub> = n(n+1) / 2

We want to find the *eighth term* of the series, which we can denote as a<sub>8</sub>.

Here's the key idea:  The eighth term (a<sub>8</sub>) is the difference between the sum of the first 8 terms (S<sub>8</sub>) and the sum of the first 7 terms (S<sub>7</sub>).

a<sub>8</sub> = S<sub>8</sub> - S<sub>7</sub>

Let's calculate S<sub>8</sub> and S<sub>7</sub> using the given formula:

*   S<sub>8</sub> = 8(8+1) / 2 = 8 * 9 / 2 = 72 / 2 = 36
*   S<sub>7</sub> = 7(7+1) / 2 = 7 * 8 / 2 = 56 / 2 = 28

Now we can find a<sub>8</sub>:

a<sub>8</sub> = S<sub>8</sub> - S<sub>7</sub> = 36 - 28 = 8

Therefore, the eighth term of the series is 8.
Was this helpful? (YES/NO): yes
Thanks I am glad it helped.
